In [1]:
import os
#Ensure directory is set to /home/marqezorr/MyTop5
ROOT_DIR = os.path.join("/home", "marqezorr", "MyTop5")
os.chdir(ROOT_DIR)
print("Current directory:", os.getcwd()) #Ensure we're in the MyTop5 directory!

Current directory: /home/marqezorr/MyTop5


In [2]:
from src.preprocess import load_anime, build_content_matrix, load_ratings, USERS_CSV, train_svd

In [3]:
#TEST TO ENSURE LOAD_ANIME WORKS PROPERLY
anime_df = load_anime("data/anime-filtered.csv")
#TEST TO ENSURE BUILD_CONTENT_MATRIX WORKS PROPERLY
matrix, index_map = build_content_matrix(anime_df)

Loading anime metadata...
14,952 anime present BEFORE cleaning
13,602 anime present AFTER cleaning
Building TF-IDF synopsis matrix...
TF-IDF matrix shape: (13602, 5000)
Now building multi-hot genre matrix...
Genre matrix shape: (13602, 42)
Unique genres found: 42
Combined feature matrix shape: (13602, 5042)


In [4]:
# Shape of the full feature matrix
print("Matrix shape:", matrix.shape)

Matrix shape: (13602, 5042)


> The matrix shape should =  [# of anime present post-cleaning], [**TFIDF_MAX_FEATURES** + # of Unique genres found]

In [5]:
# Confirm index map is working
print("First 5 entries:", list(index_map.items())[:5])

First 5 entries: [(1, 0), (5, 1), (6, 2), (7, 3), (8, 4)]


> The index map above means that anime_id 1 is located at row 0, anime_id 5 is located at row 1, etc.

In [6]:
# Check sparsity — useful to know for memory purposes
density = matrix.nnz / (matrix.shape[0] * matrix.shape[1])
print(f"Matrix density: {density:.4%}")

Matrix density: 0.5990%


> Our matrix density is extremely low, but this is to be expected (and is the reason why a csr_matrix was utilized, as this is masively more space-efficient compared to a numpy array.

In [7]:
#Test Load_ratings using our anime_df's anime_id col as our ids set
ratings_df = load_ratings(USERS_CSV, anime_ids=set(anime_df["anime_id"]))

Loading user ratings (this may take a moment)...
Raw ratings loaded: 24,325,191 rows
# of records after anime filtering: 23,808,821 rows
# of records after removing unscored entries: 23,808,821 rows
# of records after USER filter (>=50 ratings): 21,595,925 rows
# of records after ANIME filter (>=20 ratings): 21,573,340 rows
Unique USERS: 116,999
Unique ANIME: 9,108


In [ ]:
#train our SVD model using ratings_df
#WARNING - This may take a long time!
svd_model = train_svd(ratings_df)

Training our SVD model...
Hyperparameter tuning now...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.


> We can now check to see what parameter values were deemed best, and run other checks...

In [ ]:
# Confirm best parameters were found
print("Best params:", svd_model.n_factors, svd_model.n_epochs, svd_model.lr_all, svd_model.reg_all)

In [ ]:
# Confirm the model has a trainset (i.e. it was fitted successfully)
print("Trainset size:", svd_model.trainset.n_ratings)

In [ ]:
# Spot check — predict a rating for a known user/anime pair from your ratings data
sample = ratings_df.sample(1, random_state=42).iloc[0]
pred = svd_model.predict(sample["user_id"], sample["anime_id"])
print(f"Actual: {sample['rating']} | Predicted: {pred.est:.2f}")